# Collaborative Filtering Movie Recommendation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd


Reading in movie data from CSV

In [ ]:
movies_5000 = pd.read_csv('/content/drive/MyDrive/Dataset/tmdb_5000_movies.csv')


In [ ]:
movies_5000.head()


In [ ]:
movies_25M = pd.read_csv("/content/drive/MyDrive/Dataset/ml-25m/ml-25m/movies.csv")

In [ ]:
movies_25M.head(10)


Creating a cleaner title by removing the year

In [ ]:
from os import remove
import re

def clean_title(title):
   clean_title = re.sub("[^a-zA-Z0-9]", " ", title)
   remove_year = clean_title[:-5].strip()
   return remove_year

In [ ]:
movies_25M["clean_title"] = movies_25M["title"].apply(clean_title)

In [ ]:
movies_25M.head()

In [ ]:
movies_25M.drop('title', axis = 1, inplace = True)
movies_25M.head()

In [ ]:
movies_5000 = movies_5000[['title']]
movies_5000.head()

Merging the 5K dataset with the 25M dataset

In [ ]:
merged_dataset = pd.merge(movies_25M, movies_5000, left_on='clean_title', right_on='title', how='inner')
merged_dataset.drop('title', axis=1, inplace=True)

merged_dataset.head()


In [ ]:
len(merged_dataset)

Data was lost when merging both datasets:
4809 - 3728 = 1081 (amount of data missing)

Using TFIDF vectorization technique to make each title as a numerical vector based on the words in the title

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))
tfidf = vectorizer.fit_transform(merged_dataset['clean_title'])

When search for a title, the query is converted to its own vector which is then used to match with other similar vectors by using cosine similarity. Consine similarity finds the closest vectors by cosine angle

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search(title):
  query_vec = vectorizer.transform([title])
  similarity = cosine_similarity(query_vec, tfidf).flatten()
  indices = np.argpartition(similarity, -5)[-5:]
  results = merged_dataset.iloc[indices][::-1]
  return results

Search bar widget to find movie

In [ ]:
import ipywidgets as widgets
from IPython.display import display

input = widgets.Text(
    value = "Toy Story",
    description = "Movie Title: ",
    disabled = False
)
list = widgets.Output()

def on_type(data):
  with list:
    list.clear_output()
    title = data["new"]
    if len(title) > 5:
      display(search(title))

input.observe(on_type, names='value')
display(input, list)


Reading in the review data from CSV

In [ ]:
ratings_25M = pd.read_csv('/content/drive/MyDrive/Dataset/ml-25m/ml-25m/ratings.csv')

In [ ]:
new_ratings = ratings_25M[ratings_25M['movieId'].isin(set(merged_dataset['movieId']))]

In [ ]:
new_ratings.head()

In [ ]:
new_ratings.dtypes

In [ ]:
movie_id = 1

Preparing the review data by getting the people who liked similar movies

In [ ]:
same_users = new_ratings[(new_ratings['movieId'] == movie_id) & (new_ratings["rating"] > 4)]["userId"].unique()

In [ ]:
same_users

In [ ]:
same_user_recs = new_ratings[(new_ratings["userId"].isin(same_users)) & (new_ratings["rating"] > 4)]["movieId"]

In [ ]:
same_user_recs

Extracting the movies that more than 10% of users recommended

In [ ]:
same_user_recs = same_user_recs.value_counts() / len(same_users)
same_user_recs = same_user_recs[same_user_recs > .1]

In [ ]:
same_user_recs

Find people who have rated movies and rated them highly. Finding the percentage of all users who recommended the movies

In [ ]:
all_users = new_ratings[(new_ratings["movieId"]).isin(same_user_recs.index) & (new_ratings["rating"] > 4)]

In [ ]:
all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [ ]:
all_users_recs

Find the percentages of users who are similar to us liked them and how much everyone liked them

In [ ]:
rec_percentages = pd.concat([same_user_recs, all_users_recs], axis = 1)
rec_percentages.columns = ["similar", "all"]

In [ ]:
rec_percentages